# Fine-tuning Qwen2.5-VL-4B for Context Reasoning (Location)

This notebook fine-tunes Qwen2.5-VL-3B-Instruct on your WebDataset for context reasoning based on location analysis.

In [ ]:
# # Install required packages
# !pip install -q transformers accelerate peft bitsandbytes
# !pip install -q webdataset pillow
# !pip install -q trl

In [ ]:
import sys, os
sys.path.append(os.path.join(os.path.abspath(''), '..'))
import config

In [ ]:
import torch

In [ ]:
import os
import re
import glob
import json
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import webdataset as wds
import matplotlib.pyplot as plt

In [ ]:
from transformers import (
    AutoProcessor, 
    Qwen2_5_VLForConditionalGeneration,  # Qwen2.5-VL 使用这个
    BitsAndBytesConfig,
    TrainingArguments
)

from trl import SFTTrainer, SFTConfig
from datasets import Dataset, load_from_disk
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

In [ ]:
from sklearn.metrics import (
    accuracy_score, 
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

# ============================================================================
# Configuration
# ============================================================================

In [ ]:
# Get the actual tar files
base_path = config.SFT_DATA_ROOT
tar_files = sorted(glob.glob(os.path.join(base_path, 'location_webdataset_test', 'train-*.tar')))
print(f"Found {len(tar_files)} tar files:")
for f in tar_files:
    print(f"  - {f}")

CONFIG = {
    'model_id': config.QWEN_3B_MODEL,
    'webdataset_files': tar_files,
    'output_dir': './qwen25_finetuned_location_specialist',
    'batch_size': 1,
    'gradient_accumulation_steps': 16,
    'num_epochs': 3,
    'learning_rate': 2e-4,
    'max_length': 2048,  # Increased for full responses
    'warmup_ratio': 0.1,
    'logging_steps': 10,
    'save_steps': 500,
    'eval_steps': 100,
    'use_4bit': True,
    'use_lora': True,
    'dataset_size': 20000,
}

print("\nConfiguration:")
for k, v in CONFIG.items():
    if k != 'webdataset_files':
        print(f"  {k}: {v}")

In [ ]:
# Quantization config
bnb_config = None
if CONFIG['use_4bit']:
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

print(f"\nLoading model from: {CONFIG['model_id']}...")

# 改用 Qwen2VLForConditionalGeneration
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    CONFIG['model_id'],
    quantization_config=bnb_config,
    device_map="auto",
    local_files_only=True,
    torch_dtype=torch.bfloat16  # 或者改成 dtype=torch.bfloat16 避免 warning
)

processor = AutoProcessor.from_pretrained(
    CONFIG['model_id'],
    local_files_only=True
)

# Set padding token if not set
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# ============================================================================
# Location-specific Prompt Generator
# ============================================================================


In [ ]:
def generate_location_prompt(new_object):
    prompt = f"""The object to analyze is {new_object}, which is located inside the red bounding box in the image.
    Considering image quality is not a factor, do you think the object {new_object} inside the red bounding box is in-context or out-of-context?
    Important: Please assume that the object {new_object} inside the red bounding box genuinely exists in the scene, regardless of how realistic or natural it appears visually. Even if the object looks artificially added, poorly rendered, or unrealistic in terms of visual quality, treat it as if it is truly present in that location. Focus only on whether its presence makes contextual sense, not on its visual realism.
    The criterion for determining whether an object is in- or out-of-context is as follows:
    Location: Evaluate whether the object inside the red bounding box is placed in a physically and contextually reasonable position, such as being supported by a surface, on the ground, or in a plausible environment.
       If the object is floating in the air, embedded in another object, or placed in an unusual spot, it is considered out-of-context.
    Important notes:
    - The threshold is "unusual" or "uncommon", NOT "impossible". If something is uncommon or atypical in real-world contexts, it should be considered out-of-context.
    - Your summary judgment must be consistent with your analysis. If you identify something as unusual, uncommon, or abnormal in your analysis, then your summary MUST be marked as out-of-context. Conversely, if your analysis describes the situation as normal, common, or typical, then mark it as in-context.
    - Analyze only the object {new_object} inside the red bounding box.
    - Do not reinterpret or substitute the given object with another category. Even if the object visually resembles another type, always analyze it strictly as the given object {new_object}.
    - Ignore the visual quality, realism, or rendering quality of the object. Focus solely on whether its contextual presence makes sense.
    Please provide the analysis of the object according to the Location criterion, then give a summary, and finally provide your final decision.
    For example, your answer could be:
    Analysis:
    Location: The {new_object} inside the red bounding box is floating in mid-air without any support.
    Summary:
    Final decision: Out-of-context.
    Or your answer could be:
    Analysis:
    Location: The {new_object} inside the red bounding box is normally placed on the ground/surface.
    Summary:
    Final decision: In-context.
    Your answer is:"""
    
    return prompt

# ============================================================================
# Load Data and Extract Location-related Parts
# ============================================================================


In [ ]:
def extract_location_section(full_response):
    """Extract the Location-related section from the full 72B model response"""
    import re
    
    # Try to extract the Location analysis section
    location_pattern = r'Location:(.*?)(?:Summary:|$)'
    location_match = re.search(location_pattern, full_response, re.DOTALL | re.IGNORECASE)
    
    if location_match:
        location_analysis = location_match.group(1).strip()
    else:
        # Fallback: look for any location-related content
        location_analysis = "Location analysis not found in response"
    
    # Try to extract the Location conclusion from Summary
    summary_pattern = r'Summary:.*?Location:\s*(In-context|Out-of-context)'
    summary_match = re.search(summary_pattern, full_response, re.DOTALL | re.IGNORECASE)
    
    if summary_match:
        location_conclusion = summary_match.group(1)
    else:
        # Fallback: check final decision if all were same
        if "Final decision: In-context" in full_response:
            location_conclusion = "In-context"
        elif "Final decision: Out-of-context" in full_response:
            location_conclusion = "Out-of-context"
        else:
            location_conclusion = "Unknown"
    
    # Construct a focused response about location
    focused_response = f"""Location Analysis:
{location_analysis}

Location Conclusion: {location_analysis}"""
    
    return focused_response

In [ ]:
def load_webdataset_to_list(files, max_samples=None):
    """Load WebDataset and convert to list of samples"""
    dataset = wds.WebDataset(files, shardshuffle=False).decode("pil")
    
    samples = []
    for i, sample in enumerate(tqdm(dataset, desc="Loading dataset", total=max_samples)):
        if max_samples and i >= max_samples:
            break
        samples.append(sample)
    
    return samples

In [ ]:
def load_object_info():
    """Load the object information from CSV"""
    csv_path = config.TESTING_CSV

    df = pd.read_csv(csv_path)
    # Create a dictionary mapping coco_index to replacement_object
    object_map = dict(zip(df['coco_index'], df['replacement_object']))
    return object_map

In [ ]:
# Load object mapping
object_map = load_object_info()

In [ ]:
def reformat_ground_truth_response_with_label(original_response, label):
    """
    Reformat ground truth response with known label.
    
    Args:
        original_response: The original analysis text
        label: The known label ("In-context" or "Out-of-context")
    """
    
    formatted_response = f"""Analysis:
Location: {original_response.strip()}

Summary:
Final decision: {label}"""
    
    return formatted_response

In [ ]:
def format_sample_for_sft(sample, object_map, processor):
    """Format a sample for SFT training"""
    coco_index = sample['json']['coco_index']
    
    # Get object name
    object_name = object_map.get(coco_index, "object")
    
    # Generate prompt
    user_prompt = generate_location_prompt(object_name)
    
    # Get response
    original_response = sample['json']['qwen_response']
    label = sample['json']['label']  # Assuming this contains "In-context" or "Out-of-context"

    if "Analysis:" in original_response and "Summary:" in original_response and "Final decision:" in original_response:
        target_response = original_response
    else:
        # Reformat with known label
        target_response = reformat_ground_truth_response_with_label(original_response, label)
    
    # Create conversation with image
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample['bbox.png']},
                {"type": "text", "text": user_prompt}
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": target_response}
            ]
        }
    ]
    
    # Apply chat template
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    
    return {
        'text': text,
        'image_data': sample['bbox.png'],
        'label': sample['json']['label'],
        'coco_index': coco_index,
        'ground_truth_response': target_response,  # ADD THIS LINE
    }

In [ ]:
# raw_samples = load_webdataset_to_list(CONFIG['webdataset_files'], max_samples=50_000)
# print(f"✅ Loaded {len(raw_samples)} samples")

In [ ]:
cache_dir = './cached_dataset_test'
if os.path.exists(cache_dir):
    full_dataset = load_from_disk(os.path.join(cache_dir, 'test'))
    print(f"✅ Loaded from cache: {len(full_dataset)} test")
else:
    print("\n📚 Loading WebDataset...")
    raw_samples = load_webdataset_to_list(CONFIG['webdataset_files'], max_samples=CONFIG['dataset_size'])
    print(f"✅ Loaded {len(raw_samples)} samples")
    
    formatted_data = [format_sample_for_sft(sample, object_map, processor) 
                      for sample in tqdm(raw_samples)]
    
    dataset_dict = {
        'text': [d['text'] for d in formatted_data],
        'image_data': [d['image_data'] for d in formatted_data],
        'label': [d['label'] for d in formatted_data],
        'coco_index': [d['coco_index'] for d in formatted_data],
        'ground_truth_response': [d['ground_truth_response'] for d in formatted_data],
    }
    
    full_dataset = Dataset.from_dict(dataset_dict)
    
    # 保存缓存
    os.makedirs(cache_dir, exist_ok=True)
    print("💾 Saving to cache...")
    full_dataset.save_to_disk(os.path.join(cache_dir, 'test'))
    print("✅ Cached!")